# 🪨 CRISM Mineral Classification v2

**7-Branch ResBlock + CR/SNV + HierarchicalLoss + Spatial Smoothing + Attention Analysis**

| Feature | Detail |
|---|---|
| Branches | 7 (full 350-band IR, 1.02–3.92μm) |
| Preprocessing | Fill interpolation → Continuum Removal → SNV |
| Architecture | PreActResBlock + Dilated Conv + Branch Attention |
| Loss | Hierarchical (group → subgroup → class) |
| Split | Observation-wise (no scene leakage) |
| Data | 506K pixels → ~451K after class filter, 77 obs |
| Classes | Known minerals only (TAXONOMY, ~30 classes) |

---
**사용법**: Runtime → Change runtime type → **GPU (T4)** 선택 후 전체 실행

## 0. Setup & Dependencies

In [ ]:
!pip install -q seaborn scikit-learn matplotlib numpy torch

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU 없음! Runtime → Change runtime type → GPU 설정 필요")

## 1. Download Pipeline & Data

Pipeline은 항상 최신 버전으로 다운로드

In [ ]:
import urllib.request, os

# Pipeline script — ALWAYS re-download to get latest fixes
PIPELINE_URL = "https://gist.githubusercontent.com/jejuchild/fa1f2dd37c7e7c86a426ca30588c01d1/raw/crism_v2_pipeline.py"
print("Downloading latest pipeline script...")
urllib.request.urlretrieve(PIPELINE_URL, "crism_v2_pipeline.py")
print("✅ Pipeline downloaded (latest version)")

# Training data (247 MB) — skip if already exists
DATA_URL = "https://github.com/jejuchild/crism-mineral-data/releases/download/v1.0/crism_training_data_f16.npz"
DATA_FILE = "crism_training_data_f16.npz"
if not os.path.exists(DATA_FILE):
    print(f"Downloading training data (~247 MB)...")
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    sz = os.path.getsize(DATA_FILE) / 1e6
    print(f"✅ Data downloaded: {sz:.0f} MB")
else:
    sz = os.path.getsize(DATA_FILE) / 1e6
    print(f"✅ Data already exists: {sz:.0f} MB")

## 2. Quick Data Check

In [ ]:
import numpy as np

d = np.load(DATA_FILE)
X, y = d["X"], d["y"]
obs_ids = d["obs_ids"]
obs_idx = d["obs_idx"]

print(f"Spectra shape: {X.shape}  (pixels × bands)")
print(f"Labels shape:  {y.shape}")
print(f"Observations:  {len(obs_ids)}")
print(f"Data dtype:    {X.dtype} → will upcast to float32")

# Fill band check
fill_count = (X == 0.0).sum(axis=0)
bad_bands = np.where(fill_count > len(X) * 0.1)[0]
print(f"\nFill bands (>10% fill): {bad_bands.tolist()}")
print(f"Avg fill bands per pixel: {(X == 0.0).sum(axis=1).mean():.1f}/350")

# Known vs unknown classes
TAXONOMY_IDS = {1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,38,100}
known = np.isin(y, list(TAXONOMY_IDS))
print(f"\nKnown class pixels:   {known.sum():,} ({100*known.mean():.1f}%)")
print(f"Unknown class pixels: {(~known).sum():,} ({100*(~known).mean():.1f}%)")
unknown_ids = sorted(set(y[~known].tolist()))
print(f"Unknown class IDs:    {unknown_ids}")

print(f"\nClass distribution (top 15):")
ulbl, ucnt = np.unique(y, return_counts=True)
for l, c in sorted(zip(ulbl, ucnt), key=lambda x: -x[1])[:15]:
    tag = '✅' if l in TAXONOMY_IDS else '❌'
    print(f"  {tag} Class {l:>3}: {c:>8,} ({100*c/len(y):>5.2f}%)")
if len(ulbl) > 15:
    print(f"  ... and {len(ulbl)-15} more classes")

## 3. Train Model

Pipeline flow:
1. Filter to known mineral classes (remove undefined 32-39)
2. Interpolate fill bands (0.0 → linear interp from neighbors)
3. Preprocess: clip(0,1) → Continuum Removal → SNV
4. Obs-wise split (80/10/10, no scene leakage)
5. 7-branch ResBlock CNN + HierarchicalLoss

T4 기준 약 20-30분 소요 (30 epochs)

In [ ]:
# Clean previous results if re-training
import shutil
if os.path.exists('./results_v2'):
    shutil.rmtree('./results_v2')
    print('Cleaned previous results')

!python crism_v2_pipeline.py \
    --phase train \
    --npz-path crism_training_data_f16.npz \
    --out-dir ./results_v2 \
    --epochs 30 \
    --batch 512 \
    --lr 1e-3

## 4. Evaluate & Analyze

In [ ]:
!python crism_v2_pipeline.py \
    --phase analyze \
    --npz-path crism_training_data_f16.npz \
    --out-dir ./results_v2

## 5. Results

In [ ]:
import json

# Test metrics
with open("./results_v2/test_metrics.json") as f:
    metrics = json.load(f)
print("=" * 50)
print("TEST RESULTS")
print("=" * 50)
print(f"  Accuracy:    {metrics['accuracy']:.4f}")
print(f"  Kappa:       {metrics['kappa']:.4f}")
print(f"  Macro F1:    {metrics['macro_f1']:.4f}")
print(f"  Weighted F1: {metrics['weighted_f1']:.4f}")
print(f"  Test pixels: {metrics['n_test']:,}")
print(f"  Branches:    {metrics['n_branches']}")

# Previous best for comparison
print(f"\n--- vs Previous Best (epoch_attn_v9, 5-branch, pixel-random split) ---")
print(f"  Accuracy:    0.9389  (⚠️ inflated by scene leakage)")
print(f"  Kappa:       0.9300")
print(f"  Macro F1:    0.8390")
print(f"  Weighted F1: 0.9400")
print(f"\n  Note: v9 used pixel-random split (scene leakage).")
print(f"  v2 uses obs-wise split (honest). Target: Acc>0.80, Macro F1>0.50")

### 5a. Per-Class Performance

In [ ]:
import json

with open("./results_v2/classification_report.json") as f:
    report = json.load(f)

print(f"{'Class':<28} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 70)
for name, vals in report.items():
    if name in ('accuracy', 'macro avg', 'weighted avg'):
        continue
    p = vals.get('precision', 0)
    r = vals.get('recall', 0)
    f1 = vals.get('f1-score', 0)
    s = int(vals.get('support', 0))
    icon = '✅' if f1 >= 0.85 else ('⚠️' if f1 >= 0.7 else '❌')
    print(f"{icon} {name:<26} {p:>10.3f} {r:>10.3f} {f1:>10.3f} {s:>10,}")
print("-" * 70)
for name in ('macro avg', 'weighted avg'):
    vals = report[name]
    print(f"  {name:<26} {vals['precision']:>10.3f} {vals['recall']:>10.3f} {vals['f1-score']:>10.3f} {int(vals['support']):>10,}")

### 5b. Training History

In [ ]:
import json
import matplotlib.pyplot as plt

with open("./results_v2/train_history.json") as f:
    hist = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(hist["val_acc"], 'b-', lw=2)
axes[0].set_title("Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].grid(True, alpha=0.3)

axes[1].plot(hist["loss"], 'r-', lw=2)
axes[1].set_title("Training Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(True, alpha=0.3)

axes[2].plot(hist["lr"], 'g-', lw=2)
axes[2].set_title("Learning Rate")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("LR")
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Best val accuracy: {max(hist['val_acc']):.4f} (epoch {hist['val_acc'].index(max(hist['val_acc']))+1})")

## 6. Attention Analysis — Branch Pruning

어떤 spectral branch가 중요하고, 어떤 게 불필요한지 분석

In [ ]:
from IPython.display import Image, display
import os

analysis_dir = "./results_v2/analysis"

# Branch importance bar chart
if os.path.exists(f"{analysis_dir}/attention_bar.png"):
    print("Branch Importance (🔴 PRUNE / 🟡 LOW / 🟢 KEEP):")
    display(Image(f"{analysis_dir}/attention_bar.png", width=800))

# Per-class × per-branch heatmap
if os.path.exists(f"{analysis_dir}/attention_heatmap.png"):
    print("\nPer-Class × Per-Branch Attention Heatmap:")
    display(Image(f"{analysis_dir}/attention_heatmap.png", width=900))

# Distribution
if os.path.exists(f"{analysis_dir}/attention_dist.png"):
    print("\nAttention Weight Distributions:")
    display(Image(f"{analysis_dir}/attention_dist.png", width=900))

In [ ]:
import json

with open("./results_v2/analysis/attention_analysis.json") as f:
    analysis = json.load(f)

print("=" * 60)
print("PRUNING RECOMMENDATIONS")
print("=" * 60)
icons = {"PRUNE": "🔴", "LOW": "🟡", "KEEP": "🟢"}
for p in analysis["pruning"]:
    print(f"  {icons[p['verdict']]} {p['label']}: mean={p['mean']:.4f} → {p['verdict']}")

n_prune = sum(1 for p in analysis["pruning"] if p["verdict"] == "PRUNE")
n_low = sum(1 for p in analysis["pruning"] if p["verdict"] == "LOW")
if n_prune:
    kept = [p["label"] for p in analysis["pruning"] if p["verdict"] != "PRUNE"]
    print(f"\n→ {n_prune} branch(es) 제거 가능. 다음 phase: {len(kept)} branches")
    print(f"→ 유지: {', '.join(kept)}")
else:
    print(f"\n→ 모든 branch 기여 중. 제거 불필요.")
if n_low:
    low = [p["label"] for p in analysis["pruning"] if p["verdict"] == "LOW"]
    print(f"→ 주의: {', '.join(low)} — 기여도 낮음, 추가 실험 필요")

## 7. Download Results

In [ ]:
import shutil

# Zip all results
shutil.make_archive("crism_v2_results", "zip", "./results_v2")
print(f"✅ results_v2.zip created ({os.path.getsize('crism_v2_results.zip')/1e6:.1f} MB)")

# Auto-download in Colab
try:
    from google.colab import files
    files.download("crism_v2_results.zip")
except ImportError:
    print("Not in Colab — file saved locally")